# A7 IndoBERT training (GPU)
Runs package logic on hash-verified train/validation only. It must not read the locked test split. Follow `docs/indobert-training-runbook.md` before execution.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

split_drive = Path("/content/drive/MyDrive/SIPATURE/inputs/splits")

print("Folder ditemukan:", split_drive.exists())
print("Isi folder:")
for file in sorted(split_drive.iterdir()):
    print("-", file.name)

Folder ditemukan: True
Isi folder:
- split_manifest_silver_v1.json
- train_silver_v1.jsonl
- validation_silver_v1.jsonl


In [9]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

# GitHub menerima format Basic: x-access-token:<token>
credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/jodypangaribuan/hackathon.git",
        repo_dir,
    ],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."

Return code: 0

Cloning into '/content/hackathon'...



In [10]:
%cd /content/hackathon/ml
!git log --oneline -3

/content/hackathon/ml
24f140c (HEAD -> main, origin/main, origin/HEAD) feat: implement IndoBERT pipeline for aspect-based polarity and severity training
28d3844 refactor: update dataset EDA visualization labels, refine project report documentation, and refresh manifest metadata.
21e51bd feat: implement keyword and TF-IDF baselines for silver reference evaluation


In [11]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .

/content/hackathon/ml
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 134.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 131.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [12]:
import torch
import transformers
import datasets
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.7.1+cu126
Transformers: 4.53.2
Datasets: 3.6.0
Accelerate: 1.8.1
CUDA tersedia: True
GPU: Tesla T4


In [13]:
from pathlib import Path
import shutil

drive_splits = Path("/content/drive/MyDrive/SIPATURE/inputs/splits")
local_splits = Path("/content/hackathon/ml/data/splits")

local_splits.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "train_silver_v1.jsonl",
    "validation_silver_v1.jsonl",
    "split_manifest_silver_v1.json",
]

for filename in files_to_copy:
    source = drive_splits / filename
    destination = local_splits / filename

    assert source.is_file(), f"File tidak ditemukan: {source}"

    shutil.copy2(source, destination)
    print(f"Berhasil disalin: {filename} ({destination.stat().st_size:,} byte)")

print("\nIsi folder split lokal:")

for file in sorted(local_splits.iterdir()):
    print("-", file.name)

Berhasil disalin: train_silver_v1.jsonl (798,544 byte)
Berhasil disalin: validation_silver_v1.jsonl (166,813 byte)
Berhasil disalin: split_manifest_silver_v1.json (14,405 byte)

Isi folder split lokal:
- README.md
- split_manifest_silver_v1.json
- train_silver_v1.jsonl
- validation_silver_v1.jsonl


In [15]:
%cd /content/hackathon/ml
%pip install --no-deps -e .

/content/hackathon/ml
Obtaining file:///content/hackathon/ml
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sipature-ml (pyproject.toml) ... done
  Created wheel for sipature-ml: filename=sipature_ml-0.1.0-0.editable-py3-none-any.whl size=3177 sha256=5c8930ff8f6e3500ead3acdddb5581197972e5aeafc01bc5156966014ca1ff75
  Stored in directory: /tmp/pip-ephem-wheel-cache-1kwb0u5n/wheels/38/ed/47/7d6d64ef9cfba4b46e7b5732ea81e4a46aee1d5accb9c7cd88
Successfully built sipature-ml
  Attempting uninstall: sipature-ml
    Found existing installation: sipature-ml 0.1.0
    Uninstalling sipature-ml-0.1.0:
      Successfully uninstalled sipature-ml-0.1.0


In [17]:
import sys
from pathlib import Path

source_dir = Path("/content/hackathon/ml/src")

assert source_dir.is_dir(), f"Folder tidak ditemukan: {source_dir}"

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)

Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


In [18]:
import json
from pathlib import Path

from sipature_ml.indobert import verify_training_split_hashes

split_dir = Path("/content/hackathon/ml/data/splits")
manifest_path = split_dir / "split_manifest_silver_v1.json"

manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

verified_hashes = verify_training_split_hashes(
    split_dir=split_dir,
    manifest=manifest,
)

print("Validasi hash berhasil:")
for split, file_hash in verified_hashes.items():
    print(f"- {split}: {file_hash}")

print("\nLocked test pada manifest:", manifest["test_is_locked"])
print("Split valid:", manifest["validation"]["valid"])

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

In [ ]:
from pathlib import Path
from sipature_ml.config import load_config
from sipature_ml.environment import build_environment_snapshot
from sipature_ml.indobert import run_indobert_training, validate_indobert_config

config = load_config('training')
validate_indobert_config(config)
build_environment_snapshot()

In [ ]:
RUN_ID = 'REPLACE_WITH_YYYYMMDD-HHMM_indobert-silver-v1_commit'
summary = run_indobert_training(
    split_dir=Path('data/splits'),
    artifact_dir=Path('/content/drive/MyDrive/SIPATURE'),
    run_id=RUN_ID,
)
summary

In [1]:
!nvidia-smi

Sat Aug  1 09:46:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----